In [18]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.document_loaders import DirectoryLoader
import os
dir_loader= DirectoryLoader(
    "../data/pdf", 
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)
pdf_documents= dir_loader.load()
pdf_documents
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """
    Split documents into smaller chunks for better RAG performance.
    
    Parameters:
    - chunk_size: Maximum characters per chunk (adjust based on your LLM)
    - chunk_overlap: Characters to overlap between chunks (preserves context)
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Each chunk: ~1000 characters
        chunk_overlap=chunk_overlap, # 200 chars overlap for context
        length_function=len, # How to measure length
        separators=["\n\n", "\n", " ", ""] # Split hierarchy
    )
    # Actually split the documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show what a chunk looks like
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_documents(pdf_documents)
chunks
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import Any, List, Tuple, Dict
from sklearn.metrics.pairwise import cosine_similarity
class EmbeddingManager:
    def __init__(self, model_name: str ="all-MiniLM-L6-v2"):
        self.model_name= model_name
        self.model= None
        self._load_model()
    def _load_model(self):
        try:
            print(f"Loading Embedding Model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str])->np.ndarray:
        if not self.model: 
            raise ValueError("Model not laoded")
        print(f"Generating Embeddings for {len(texts)} texts...")
        embeddings= self.model.encode(texts, show_progress_bar=True)
        print(f"Generate Embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str= "../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory= persist_directory
        self.client= None
        self.collection= None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection= self.client.get_or_create_collection(
                name=self.collection_name,
                metadata= {'description':'PDF document embeddings for RAG'}
            )
            print(f"Vector stored initialized. Collection: {self.collection_name}")
            print(f"Existing document int colelction: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents : List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must match number embeddings")
        print(f"Adding {len(documents)} documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id= f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata= dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        try:
            self.collection.add(
                ids=ids,
                embeddings= embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"successful added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store {e}")
            raise
vectorstore=VectorStore()
vectorstore
chunks
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager= embedding_manager
    def retrieve(self, query: str, top_k:int =5, score_threshold: float=0.0)->List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]
        try:
            results= self.vector_store.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas= results['metadatas'][0]
                distances= results['distances'][0]
                ids= results['ids'][0]
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score= 1-distance
                    print("DEBUG:", similarity_score, distance)
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retieval {e}")
            return []
rag_retriever=RAGRetriever(vectorstore, embedding_manager)

rag_retriever
rag_retriever.retrieve("tell me about Operating System")
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.environ.get("GROQ_API_KEY")
llm= ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)
def rag_simple(query, retriever, llm, top_k=3):
    results= retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    prompt=f"""
        context:
        {context}

        Question: {query}


        Answer:"""
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content
    
answer=rag_simple("what is Operating System? answer in your own words do not copy paste", rag_retriever, llm)
print(answer)

Split 1 documents into 1 chunks

Example chunk:
Content: Operating System: OS
is system software that manages computer hardware and software resources.
It acts as a bridge between the user and the computer hardware.
The OS provides essential services like f...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-17T22:27:47+00:00', 'source': '..\\data\\pdf\\OperatingSystem.pdf', 'file_path': '..\\data\\pdf\\OperatingSystem.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-07-17T22:27:47+00:00', 'trapped': '', 'modDate': "D:20260717222747+00'00'", 'creationDate': "D:20260717222747+00'00'", 'page': 0}
Loading Embedding Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2514.28it/s]
C:\Users\jasmi\AppData\Local\Temp\ipykernel_5820\2767073626.py:60: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


Model Loaded Successfully. Embedding dimension: 384
Vector stored initialized. Collection: pdf_documents
Existing document int colelction: 2
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.89it/s]


Generate Embeddings with shape: (1, 384)
Adding 1 documents to vector store
successful added 1 documents to vector store
Total documents in collection: 3
Retrieving documents for query: 'tell me about Operating System'
Top K: 5, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.87it/s]


Generate Embeddings with shape: (1, 384)
DEBUG: 0.5973024070262909 0.4026975929737091
DEBUG: 0.5973024070262909 0.4026975929737091
DEBUG: 0.5973024070262909 0.4026975929737091
Retrieved 3 documents (after filtering)
Retrieving documents for query: 'what is Operating System? answer in your own words do not copy paste'
Top K: 3, Score threshold: 0.0
Generating Embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 78.13it/s]


Generate Embeddings with shape: (1, 384)
DEBUG: 0.4033806324005127 0.5966193675994873
DEBUG: 0.4033806324005127 0.5966193675994873
DEBUG: 0.4033806324005127 0.5966193675994873
Retrieved 3 documents (after filtering)
An Operating System (OS) is a type of software that acts as a middleman between a computer's hardware and the user. It manages the computer's resources, such as memory and storage, and provides essential services like file management, security, and multitasking. The OS also controls the input and output devices, ensuring smooth communication between the user and the computer. It provides a user-friendly interface, either graphical or command-line, making it possible for users to interact with the computer and perform various tasks efficiently.


In [6]:
import sys
print(sys.executable)

c:\Users\jasmi\anaconda3\envs\tf_env\python.exe


In [9]:
import sys
!{sys.executable} -m pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers faiss-cpu chromadb langchain-groq python-dotenv

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 4.2 MB/s eta 0:00:05
   --- ------------------------------------ 1.6/19.8 MB 4.0 MB/s eta 0:00:05
   ---- ----------------------------------- 2.4/19.8 MB 3.9 MB/s eta 0:00:05
   ------ --------------------------------- 3.4/19.8 MB 3.9 MB/s eta 0:00:05
   -------- ------------------------------- 4.2/19.8 MB 4.0 MB/s eta 0:00:04
   ---------- ----------------------------- 5.0/19.8 MB 4.0 MB/s eta 0:00:04
   ----------- ---------------------------- 5.8/19.8 MB 4.0 MB/s eta 0:00:04
   ------------- -------------------------- 6.6/19.8 MB 3.9 MB/s eta 0:00:04
   --------------- ------------------------ 7.6/19.8 MB 3.9 MB/s eta 0:00:04
   ---------------- ----------------------- 8.4/19.8 MB 3.9 MB/s eta 0:00:03
   ------------------ --------------------- 9.2/19.8 MB 3.9 MB/s eta 0:00:03
   -------------------- ------------------- 10.0/19.8 MB 4.0 MB/s eta 0:00:03
   --

In [13]:
with open(r"../data/pdf/OperatingSystem.pdf", "rb") as f:
    header = f.read(10)
print(header)

b'An operati'


In [14]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(r"../data/pdf/OperatingSystem.pdf")
docs = loader.load()
print(len(docs))

invalid pdf header: b'opera'
EOF marker not found


PdfStreamError: Stream has ended unexpectedly